# ARGOS-MUKHTAR — Qwen3-8B Teacher

Отдельный Colab runtime для сильного LLM-учителя. Не трогает BioCore notebook.


In [ ]:
import os, sys, subprocess, torch
print('Python:', sys.version)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    free,total=torch.cuda.mem_get_info()
    print('GPU:',torch.cuda.get_device_name(0))
    print('VRAM free/total GiB:',round(free/2**30,2),round(total/2**30,2))
    assert total/2**30 >= 14.0, 'Expected a T4-class GPU or better'


In [ ]:
import subprocess, sys
pkgs=['transformers>=4.51.0','accelerate>=1.4.0','autoawq','sentencepiece']
subprocess.run([sys.executable,'-m','pip','install','-q','-U',*pkgs],check=True)
print('dependencies ready')


In [ ]:
import gc, torch
from transformers import AutoTokenizer, AutoModelForCausalLM
MODEL_ID='Qwen/Qwen3-8B-AWQ'
tokenizer=AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
teacher=AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map='auto',
    torch_dtype='auto',
    low_cpu_mem_usage=True,
    trust_remote_code=True,
)
teacher.eval()
free,total=torch.cuda.mem_get_info()
print('loaded:',MODEL_ID)
print('VRAM free GiB:',round(free/2**30,2))


In [ ]:
def ask_teacher(prompt, max_new_tokens=512, thinking=True):
    messages=[
      {'role':'system','content':'Ты сильный исследовательский учитель ARGOS-MUKHTAR. Анализируй состояние точно и выбирай действие без бессмысленных повторов.'},
      {'role':'user','content':prompt},
    ]
    text=tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
        enable_thinking=thinking,
    )
    inputs=tokenizer(text,return_tensors='pt').to(teacher.device)
    with torch.inference_mode():
        out=teacher.generate(
            **inputs,max_new_tokens=max_new_tokens,
            do_sample=True,temperature=0.6,top_p=0.95,top_k=20,
        )
    return tokenizer.decode(out[0][inputs['input_ids'].shape[1]:],skip_special_tokens=True)

print(ask_teacher('Ответь одним предложением: какую роль ты выполняешь в ARGOS-MUKHTAR?',128,False))


In [ ]:
# ARC smoke test
test_prompt='''Игра r11l: нужно выбрать объект и переместить его к видимой цели.\nНе повторяй безрезультатные клики.\nОтветь только JSON {"x":N,"y":M}.\nДемо-состояние: цель приблизительно справа-сверху от выбранного объекта.'''
print(ask_teacher(test_prompt,128,False))


## Следующий шаг
Подключить `arc3/llm_teacher_r11l_v2.py` к локальной функции `ask_teacher()` и генерировать несколько победных траекторий для DAN-gated обучения BioCore.
